## Импорт библиотек и настройка окружения

In [147]:
import pandas as pd

from utilities.toolbox import (
    dataset_audit, make_settings, SALES_DICT, SALES_DICT_PATH, TRANSACTIONS, TRANSACTIONS_PATH
)

make_settings()

Настройки отображения установлены.


## Загрузка датасетов из `raw`

In [161]:
# Читаем CSV
transactions = pd.read_csv(TRANSACTIONS_PATH)
sales_dict = pd.read_csv(SALES_DICT_PATH)

# Копируем датафреймы, чтобы не работать с источниками
transactions_copy = transactions.copy()
sales_dict_copy = sales_dict.copy()

# Приводим название колонок к нижнему регистру (snake_case)
transactions_copy.columns = (
    transactions_copy.columns
    .str.lower()
    .str.strip()
)
sales_dict_copy.columns = (
    sales_dict_copy.columns
    .str.lower()
    .str.strip()
)

## Первичный аудит данных

Проверяем `raw/Sales_transactions_2022_2025.csv`

In [162]:
dataset_audit(transactions_copy, TRANSACTIONS)

Data Quality Check: Sales_transactions_2022_2025.csv

1. Общая информация (.info()):
<class 'pandas.DataFrame'>
RangeIndex: 18045 entries, 0 to 18044
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_id        18045 non-null  str    
 1   order_id              18045 non-null  str    
 2   customer_id           18045 non-null  str    
 3   customer_name         17937 non-null  str    
 4   customer_age          17827 non-null  float64
 5   customer_gender       17883 non-null  str    
 6   customer_segment      18045 non-null  str    
 7   order_date            18045 non-null  str    
 8   order_time            18045 non-null  str    
 9   sales_channel         18045 non-null  str    
 10  store_id              18045 non-null  str    
 11  store_name            18045 non-null  str    
 12  country               18045 non-null  str    
 13  region                18045 non-null  str    
 

,transaction_id,order_id,customer_id,customer_name,customer_age,customer_gender,customer_segment,order_date,order_time,sales_channel,store_id,store_name,country,region,city,product_id,product_name,product_category,product_subcategory,quantity,unit_price,discount_percentage,sales_amount,cost_amount,profit,payment_method,order_status,shipping_method,delivery_days,return_flag,return_reason,sales_representative,promotion_code,customer_rating,inventory_level,order_year
0,T02016694,O0116694,C01482,Sophia Smith,28.00,Male,Consumer,2022-10-06,11:30:55,Online,LA-01,Los Angeles Downtown,United States,West,LOS ANGELES,P026,Webcam HD,Electronics,Cameras,2,85.29,5.00,162.05,115.61,46.44,Credit Card,Completed,Standard,4.00,No,NaN,James Anderson,SAVE10,4.00,81.00,2022
1,T02016374,O0108187,C01591,Hannah King,46.00,Female,Consumer,2024-06-03,17:06:29,Online,SYD-01,Sydney CBD,Australia,New South Wales,Sydney,P005,Office Keyboard,Electronics,Accessories,6,58.85,15.00,300.13,237.00,63.13,Debit Card,Completed,Standard,6.00,No,NaN,Ella Lewis,NaN,5.00,123.00,2024
2,T02004099,O0104099,C04140,Emma Wilson,42.00,Male,Consumer,2024-12-19,08:22:11,Online,PAR-01,Paris Central,France,Île-de-France,Paris,P017,Copy Paper Box,Office Supplies,Paper,2,43.92,15.00,74.66,73.17,1.49,Credit Card,Shipped,Standard,4.00,No,NaN,Liam Davis,NaN,4.00,93.00,2024


Проверяем `raw/sales_data_dictionary.csv`

In [163]:
dataset_audit(sales_dict_copy, SALES_DICT)

Data Quality Check: sales_data_dictionary.csv

1. Общая информация (.info()):
<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   column_name       36 non-null     str  
 1   description       36 non-null     str  
 2   data_type         36 non-null     str  
 3   example_value     36 non-null     str  
 4   business_meaning  36 non-null     str  
dtypes: str(5)
memory usage: 1.5 KB

2. Пропуски: Не обнаружены.

3. Дубликаты строк: 0

4. Пример данных (.head(3)):


,column_name,description,data_type,example_value,business_meaning
0,Transaction_ID,Unique identifier for an individual transactio...,Text,T20001234,Transaction-level key
1,Order_ID,Identifier for the customer order; multiple li...,Text,O1001234,Groups order lines into orders
2,Customer_ID,Identifier for the customer,Text,C01234,Supports repeat-customer analysis


## Обработка данных
Датасет "сырой": есть дубликаты, пропуски и поля, где пропуски являются нормой, а не ошибкой.

#### Удаление дубликатов

In [164]:
# Удаление дубликатов
before = len(transactions_copy)
transactions_copy = transactions_copy.drop_duplicates()
after = len(transactions_copy)

print(f'Удалено дубликатов: {before - after}')
print(f'Осталось строк: {after} (было: {before})')

Удалено дубликатов: 45
Осталось строк: 18000 (было: 18045)


#### Преобразование

In [165]:
# order_date и order_time сейчас строки, нужно преобразовать
transactions_copy['order_date'] = pd.to_datetime(
    transactions_copy['order_date'],
    errors='coerce',
)

transactions_copy['order_time'] = pd.to_datetime(
    transactions_copy['order_time'],
    format='%H:%M:%S',
    errors='coerce',
).dt.time

print('Преобразование дат произведено.')

Преобразование дат произведено.


In [166]:
# Преобразование возраста
transactions_copy['customer_age'] = transactions_copy['customer_age'].astype('Int64')

# Преобразование дней доставки
transactions_copy['delivery_days'] = transactions_copy['delivery_days'].astype('Int64')

print('Преобразование возраста и дней доставки произведено.')

Преобразование возраста и дней доставки произведено.


In [167]:
# Преобразование категорий метода платежа (разные категории)
# Убираем разницу между категориями типа "Apple Pay" и "apple pay"
transactions_copy['payment_method'] = (
    transactions_copy['payment_method']
    .str.lower()
)

# Заменяем 'debitcard' на 'debit card' (с пробелом)
transactions_copy['payment_method'] = transactions_copy['payment_method'].replace({
    'debitcard': 'debit card',
})

# Заполняем Nan
transactions_copy['payment_method'] = transactions_copy['payment_method'].fillna('Unknown')

In [175]:
# Преобразуем названия колонок в словаре, чтобы бились с теми, которые в новом датафрейме с транзакциями
sales_dict_copy['column_name'] = (
    sales_dict_copy['column_name']
    .str.lower()
)

In [168]:
print('Отрицательная выручка:', (transactions_copy['sales_amount'] < 0).sum())
print('Отрицательная прибыль:', (transactions_copy['profit'] < 0).sum())
print('Нулевое или отрицательное количество:', (transactions_copy['quantity'] <= 0).sum())
print('Отрицательная цена:', (transactions_copy['unit_price'] < 0).sum())
print('Скидка вне диапазона 0-100:', (transactions_copy['discount_percentage'] < 0) | (transactions_copy['discount_percentage'] > 100).sum())

Отрицательная выручка: 0
Отрицательная прибыль: 231
Нулевое или отрицательное количество: 8
Отрицательная цена: 0
Скидка вне диапазона 0-100: 0        True
1        True
2        True
3        True
4        True
         ... 
18040    True
18041    True
18042    True
18043    True
18044    True
Name: discount_percentage, Length: 18000, dtype: bool


In [169]:
# Импортируем в файл аномалии с данными, где количество товаров <= 0
transactions_copy[(transactions_copy['quantity'] <= 0)].to_csv('raw/quantity_anomalies.csv', index=False)

# Убираем данные, где количество товаров <= 0
transactions_copy = transactions_copy[(transactions_copy['quantity'] > 0)]

print('Нулевое или отрицательное количество:', (transactions_copy['quantity'] <= 0).sum())

Нулевое или отрицательное количество: 0


In [170]:
print('Отрицательная выручка:', (transactions_copy['sales_amount'] < 0).sum())
print('Отрицательная прибыль:', (transactions_copy['profit'] < 0).sum())
print('Нулевое или отрицательное количество:', (transactions_copy['quantity'] <= 0).sum())
print('Отрицательная цена:', (transactions_copy['unit_price'] < 0).sum())
print('Скидка вне диапазона 0-100:', (transactions_copy['discount_percentage'] < 0) | (transactions_copy['discount_percentage'] > 100).sum())

Отрицательная выручка: 0
Отрицательная прибыль: 231
Нулевое или отрицательное количество: 0
Отрицательная цена: 0
Скидка вне диапазона 0-100: 0        True
1        True
2        True
3        True
4        True
         ... 
18040    True
18041    True
18042    True
18043    True
18044    True
Name: discount_percentage, Length: 17992, dtype: bool


In [171]:
result = transactions_copy[(transactions_copy['profit'] < 0)]
result

,transaction_id,order_id,customer_id,customer_name,customer_age,customer_gender,customer_segment,order_date,order_time,sales_channel,store_id,store_name,country,region,city,product_id,product_name,product_category,product_subcategory,quantity,unit_price,discount_percentage,sales_amount,cost_amount,profit,payment_method,order_status,shipping_method,delivery_days,return_flag,return_reason,sales_representative,promotion_code,customer_rating,inventory_level,order_year
140,T02005486,O0102743,C03782,Charlotte King,28,Male,Consumer,2025-11-10,14:41:18,Retail Store,MAN-01,Manchester North,United Kingdom,England,Manchester,P017,Copy Paper Box,Office Supplies,Paper,1,41.83,20.00,33.47,34.11,-0.65,debit card,Completed,Pickup,0,No,NaN,Daniel Miller,SPRING15,5.00,43.00,2025
353,T02004998,O0104998,C04913,Olivia Davis,33,Female,Corporate,2023-05-21,16:27:44,B2B Portal,LON-01,London City,United Kingdom,England,London,P017,Copy Paper Box,Office Supplies,Paper,4,45.85,15.00,155.90,155.97,-0.07,debit card,Completed,Standard,4,No,NaN,Ella Lewis,BLACKFRIDAY25,4.00,60.00,2023
462,T02003087,O0103087,C02292,Ava Johnson,18,Male,Consumer,2024-12-30,16:58:11,Retail Store,MUC-01,Munich Zentrum,Germany,Bavaria,Munich,P017,Copy Paper Box,Office Supplies,Paper,3,43.27,20.00,103.84,104.08,-0.24,apple pay,Shipped,Standard,6,No,NaN,Henry White,NaN,4.00,75.00,2024
616,T02009907,O0109907,C03188,William Smith,18,Male,Corporate,2023-11-13,16:13:03,B2B Portal,TOR-01,Toronto Central,Canada,Ontario,Toronto,P016,Laser Toner,Office Supplies,Consumables,4,113.45,25.00,340.35,348.42,-8.07,cash,Completed,Pickup,0,No,NaN,Michael Clark,NaN,NaN,53.00,2023
792,T02007661,O0103830,C02839,James Johnson,41,Female,Consumer,2023-12-05,15:20:54,Retail Store,LON-01,London City,United Kingdom,England,London,P017,Copy Paper Box,Office Supplies,Paper,1,45.80,20.00,36.64,37.03,-0.38,credit card,Completed,Pickup,0,No,NaN,Daniel Miller,NaN,NaN,59.00,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17860,T02010239,O0110239,C03403,Max Moore,33,Male,Corporate,2025-12-13,15:22:49,B2B Portal,MAN-01,Manchester North,United Kingdom,England,Manchester,P016,Laser Toner,Office Supplies,Consumables,9,121.72,30.00,766.85,852.92,-86.07,credit card,Completed,Standard,2,No,NaN,Emma Carter,HOLIDAY20,4.00,31.00,2025
17866,T02003306,O0101653,C01007,Isla Taylor,56,Male,Consumer,2025-11-01,19:45:38,Online,AUS-01,Austin Business,United States,South,Austin,P003,EdgeTab 11 Tablet,Electronics,Tablets,1,523.96,25.00,392.97,429.15,-36.18,debit card,Completed,Next Day,4,No,NaN,Amelia Jackson,HOLIDAY20,5.00,70.00,2025
17904,T02014006,O0114006,C05459,Isla Jones,33,Male,Consumer,2024-11-14,16:40:54,Retail Store,MUC-01,Munich Zentrum,Germany,Bavaria,Munich,P017,Copy Paper Box,Office Supplies,Paper,3,44.18,20.00,106.03,106.92,-0.89,credit card,Shipped,Pickup,0,No,NaN,Olivia Martin,NaN,1.00,64.00,2024
17968,T02011242,O0105621,C03869,Emily Green,41,Female,Returning Customer,2025-12-15,09:20:23,Retail Store,AUS-01,Austin Business,United States,South,Austin,P017,Copy Paper Box,Office Supplies,Paper,3,42.73,25.00,96.14,100.87,-4.73,paypal,Completed,Standard,7,No,NaN,Daniel Miller,NaN,4.00,58.00,2025


In [181]:
sales_dict_copy.head(100)

,column_name,description,data_type,example_value,business_meaning
0,transaction_id,Unique identifier for an individual transactio...,Text,T20001234,Transaction-level key
1,order_id,Identifier for the customer order; multiple li...,Text,O1001234,Groups order lines into orders
2,customer_id,Identifier for the customer,Text,C01234,Supports repeat-customer analysis
3,customer_name,Customer name from CRM,Text,Emma Carter,Customer display/identification
4,customer_age,Customer age at transaction,Integer,42,Demographic analysis; rare anomalies included
5,customer_gender,Customer-reported gender,Text,Female,Demographic segmentation
6,customer_segment,Commercial/lifecycle customer segment,Text,Returning Customer,Customer behavior and value analysis
7,order_date,Date transaction was placed,Date,2024-11-29,Time-series and seasonality analysis
8,order_time,Time transaction was placed,Time,14:37:21,Hourly and anomaly analysis
9,sales_channel,Order channel,Text,Online,Channel performance
